# 03. Pipeline de Modelos de Predicción
## TFM — Estimación de Precios de la Electricidad y Arbitraje Energético
### Jaime Cremades Castelló | CUNEF Universidad | 2026

Entrenamiento y evaluación de los siete modelos de predicción de precios eléctricos.

**Modelos implementados:**
- SARIMA — baseline estadístico con ventana deslizante de 30 días
- XGBoost — 24 modelos independientes por hora, GridSearch 81 combinaciones
- LSTM — red recurrente multivariante, GridSearch 36 combinaciones
- N-BEATS — red de expansión de bases univariante, implementación propia en PyTorch
- NBEATSx — extensión multivariante de N-BEATS con loader mixto, implementación propia en PyTorch
- BiGRU-KAN — red recurrente bidireccional con capas KAN, GridSearch 36 combinaciones
- BNN con MC Dropout — red bayesiana probabilística, 24 modelos por hora

**Input:** `dataset_final_clean.csv`
**Output:** `preds_*.csv`, `precios_reales_test.csv`, `predicciones_test.csv`

**Entorno recomendado:** Google Colab Pro con GPU A100


# TFM — Modelos de Predicción de Precios Eléctricos
## Jaime Cremades Castelló | CUNEF Universidad
### Notebook completo — Google Colab Pro

**Instrucciones:**
1. Activa GPU: Entorno de ejecución → Cambiar tipo de entorno → GPU
2. Sube el dataset a Google Drive en la carpeta TFM
3. Ejecuta desde la Sección 0 en orden

**Modelos implementados:**
- SARIMA — baseline estadístico, orden (2,1,0)(2,0,0,24), ventana deslizante de 30 días
- XGBoost — baseline ML, GridSearch 81 combinaciones, 24 modelos por hora
- LSTM — red recurrente multivariante, GridSearch 36 combinaciones
- N-BEATS — red de expansión de bases univariante, GridSearch 36 combinaciones
- NBEATSx — extensión multivariante de N-BEATS con 25 variables exógenas, via NeuralForecast
- BiGRU-KAN — red recurrente bidireccional con capas KAN, GridSearch 36 combinaciones
- BNN con MC Dropout — red bayesiana probabilística, GridSearch 36 combinaciones x 24 horas


---
## 0. Configuración e imports

In [1]:
from google.colab import drive
drive.mount('/content/drive')
print('Drive montado')

Mounted at /content/drive
Drive montado


In [2]:
# Instalación de dependencias
# neuralforecast se instala solo si se va a entrenar NBEATSx por primera vez
!pip install pmdarima -q
!pip install neuralforecast -q
print('Dependencias instaladas')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 689.1/689.1 kB 41.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 287.0/287.0 kB 21.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 348.2/348.2 kB 36.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 831.6/831.6 kB 60.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.8/73.8 MB 14.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 447.2/447.2 kB 39.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.6/46.6 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 419.5/419.5 kB 34.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.5/87.5 kB 9.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 53.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requ

In [3]:
import os
import random
import pickle
import joblib
import itertools
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
warnings.filterwarnings('ignore')

from sklearn.preprocessing import MinMaxScaler
from xgboost import XGBRegressor

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

# ── Rutas ──────────────────────────────────────────────────────────────────────
DRIVE_FOLDER = '/content/drive/MyDrive/TFM'
DATA_PATH    = f'{DRIVE_FOLDER}/dataset_model_ready.csv'
os.makedirs(DRIVE_FOLDER, exist_ok=True)

# ── Dispositivo ────────────────────────────────────────────────────────────────
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Dispositivo: {DEVICE}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
else:
    print('ADVERTENCIA: No se detecta GPU. Activa GPU en Entorno de ejecucion.')

# ── Hiperparámetros globales ───────────────────────────────────────────────────
HORIZON    = 24     # horas a predecir
WINDOW     = 168    # ventana de entrada (7 dias)
BATCH_SIZE = 64
TARGET     = 'price_spain'

FEATURES = [
    'hour', 'dayofweek', 'month',
    'Prevista', 'error_demanda',
    'hidraulica_generation', 'nuclear_generation', 'ciclo_combinado_generation',
    'net_imports', 'bombeo_neto',
    'renovable_ratio', 'nuclear_ratio',
    'hour_sin', 'hour_cos',
    'price_lag_1', 'price_lag_24',
    'is_weekend', 'is_holiday',
    'gas_price', 'co2_price',
    'wind_generation_forecast', 'solar_generation_forecast',
    'temperature_forecast', 'wind_speed_forecast',
    'base_vs_demanda'
]
N_FEATURES = len(FEATURES)

# ── Espacio de busqueda GridSearch (redes neuronales) ─────────────────────────
PARAM_SPACE = {
    'hidden_size': [64, 128, 256],
    'num_layers':  [1, 2],
    'dropout':     [0.1, 0.2, 0.3],
    'lr':          [0.001, 0.0001]
}
ALL_COMBINATIONS = [
    {'hidden_size': h, 'num_layers': n, 'dropout': d, 'lr': l}
    for h, n, d, l in itertools.product(
        PARAM_SPACE['hidden_size'], PARAM_SPACE['num_layers'],
        PARAM_SPACE['dropout'],    PARAM_SPACE['lr']
    )
]
print(f'Combinaciones GridSearch redes neuronales: {len(ALL_COMBINATIONS)}')
print('Imports OK')

Dispositivo: cuda
GPU: Tesla T4
Combinaciones GridSearch redes neuronales: 36
Imports OK


---
## 1. Carga y preprocesamiento

In [4]:
# Carga del dataset
df = pd.read_csv(DATA_PATH, parse_dates=['datetime'])
df = df.sort_values('datetime').reset_index(drop=True)
print(f'Dataset: {df.shape[0]} registros')
print(f'Periodo: {df["datetime"].min()} — {df["datetime"].max()}')

# Split temporal cronologico sin data leakage
train_mask = df['datetime'].dt.year <= 2023
val_mask   = df['datetime'].dt.year == 2024
test_mask  = df['datetime'].dt.year >= 2025

df_train = df[train_mask].copy()
df_val   = df[val_mask].copy()
df_test  = df[test_mask].copy()

print(f'Train: {df_train.shape[0]} | Val: {df_val.shape[0]} | Test: {df_test.shape[0]}')

Dataset: 53585 registros
Periodo: 2020-01-01 23:00:00+00:00 — 2026-02-28 22:00:00+00:00
Train: 34773 | Val: 8741 | Test: 10071


In [5]:
# Normalizacion [-1, 1] ajustada solo sobre train para evitar data leakage
scaler_X = MinMaxScaler(feature_range=(-1, 1))
scaler_y = MinMaxScaler(feature_range=(-1, 1))

X_train = scaler_X.fit_transform(df_train[FEATURES])
y_train = scaler_y.fit_transform(df_train[[TARGET]])
X_val   = scaler_X.transform(df_val[FEATURES])
y_val   = scaler_y.transform(df_val[[TARGET]])
X_test  = scaler_X.transform(df_test[FEATURES])
y_test  = scaler_y.transform(df_test[[TARGET]])

y_train_real = df_train[TARGET].values
y_val_real   = df_val[TARGET].values
y_test_real  = df_test[TARGET].values

# Guardar scalers para el notebook de arbitraje
joblib.dump(scaler_X, f'{DRIVE_FOLDER}/scaler_X.pkl')
joblib.dump(scaler_y, f'{DRIVE_FOLDER}/scaler_y.pkl')

# Guardar precios reales del test
pd.DataFrame({
    'datetime': df_test['datetime'].values,
    'precio_real': y_test_real
}).to_csv(f'{DRIVE_FOLDER}/precios_reales_test.csv', index=False)

print('Scalers y precios reales guardados')

# Funcion de desnormalizacion
def desnormalizar(y_norm):
    if isinstance(y_norm, torch.Tensor):
        y_norm = y_norm.numpy()
    if y_norm.ndim == 1:
        y_norm = y_norm.reshape(-1, 1)
    return scaler_y.inverse_transform(y_norm).flatten()

# Funcion de metricas
# MAPE se calcula solo sobre horas con precio absoluto > 10 EUR/MWh
# para evitar inestabilidad numerica con precios cercanos a cero o negativos
def calcular_metricas(y_real, y_pred, nombre=''):
    mae  = np.mean(np.abs(y_real - y_pred))
    rmse = np.sqrt(np.mean((y_real - y_pred)**2))
    mask = np.abs(y_real) > 10
    mape = np.mean(np.abs((y_real[mask] - y_pred[mask]) / y_real[mask])) * 100
    if nombre:
        print(f'{nombre}: MAE={mae:.4f} | RMSE={rmse:.4f} | MAPE={mape:.4f}%')
    return {'MAE': mae, 'RMSE': rmse, 'MAPE': mape}

resultados = {}
print('Preprocesamiento completado')

Scalers y precios reales guardados
Preprocesamiento completado


In [6]:
# Construccion de ventanas temporales para modelos secuenciales
def crear_ventanas(X, y, window=WINDOW, horizon=HORIZON):
    """Ventanas multivariantes para LSTM y BiGRU-KAN."""
    X_seq, y_seq = [], []
    for i in range(len(X) - window - horizon + 1):
        X_seq.append(X[i:i+window])
        y_seq.append(y[i+window:i+window+horizon].flatten())
    return np.array(X_seq), np.array(y_seq)

def crear_ventanas_univariante(y, window=WINDOW, horizon=HORIZON):
    """Ventanas univariantes para N-BEATS (solo precio)."""
    X_seq, y_seq = [], []
    for i in range(len(y) - window - horizon + 1):
        X_seq.append(y[i:i+window])
        y_seq.append(y[i+window:i+window+horizon])
    return np.array(X_seq), np.array(y_seq)

def crear_ventanas_mixtas(y_norm, X_norm, window=WINDOW, horizon=HORIZON):
    """
    Ventanas mixtas para NBEATSx.
    X_price: precio historico normalizado (window,)
    X_exog:  variables exogenas de las proximas 24h (horizon, n_features)
    y:       precio de las proximas 24h (horizon,)
    """
    X_price, X_exog, y_out = [], [], []
    for i in range(len(y_norm) - window - horizon + 1):
        X_price.append(y_norm[i:i+window])
        X_exog.append(X_norm[i+window:i+window+horizon])
        y_out.append(y_norm[i+window:i+window+horizon])
    return np.array(X_price), np.array(X_exog), np.array(y_out)

X_train_seq, y_train_seq = crear_ventanas(X_train, y_train)
X_val_seq,   y_val_seq   = crear_ventanas(X_val,   y_val)
X_test_seq,  y_test_seq  = crear_ventanas(X_test,  y_test)

y_train_norm = y_train.flatten()
y_val_norm   = y_val.flatten()
y_test_norm  = y_test.flatten()

X_nbeats_train, y_nbeats_train = crear_ventanas_univariante(y_train_norm)
X_nbeats_val,   y_nbeats_val   = crear_ventanas_univariante(y_val_norm)
X_nbeats_test,  y_nbeats_test  = crear_ventanas_univariante(y_test_norm)

X_price_train, X_exog_train, y_mix_train = crear_ventanas_mixtas(y_train_norm, X_train)
X_price_val,   X_exog_val,   y_mix_val   = crear_ventanas_mixtas(y_val_norm,   X_val)
X_price_test,  X_exog_test,  y_mix_test  = crear_ventanas_mixtas(y_test_norm,  X_test)

print(f'Ventanas multivariantes train: {X_train_seq.shape}')
print(f'Ventanas univariantes train:   {X_nbeats_train.shape}')
print(f'Ventanas mixtas train:         X_price {X_price_train.shape} | X_exog {X_exog_train.shape}')

Ventanas multivariantes train: (34582, 168, 25)
Ventanas univariantes train:   (34582, 168)
Ventanas mixtas train:         X_price (34582, 168) | X_exog (34582, 24, 25)


In [7]:
# Dataset PyTorch y DataLoaders
class SerieDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.FloatTensor(X)
        self.y = torch.FloatTensor(y)
    def __len__(self):
        return len(self.X)
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

class SerieDatasetMixto(Dataset):
    """
    Dataset para NBEATSx — loader mixto.
    X_price: ventana univariante de precio historico (batch, window)
    X_exog:  variables exogenas de las 24 horas a predecir (batch, horizon, n_features)
    y:       precio de las 24 horas a predecir (batch, horizon)
    """
    def __init__(self, X_price, X_exog, y):
        self.X_price = torch.FloatTensor(X_price)
        self.X_exog  = torch.FloatTensor(X_exog)
        self.y       = torch.FloatTensor(y)
    def __len__(self):
        return len(self.y)
    def __getitem__(self, idx):
        return self.X_price[idx], self.X_exog[idx], self.y[idx]

# Loader univariante — N-BEATS
train_nbeats = DataLoader(SerieDataset(X_nbeats_train, y_nbeats_train), batch_size=BATCH_SIZE, shuffle=False)
val_nbeats   = DataLoader(SerieDataset(X_nbeats_val,   y_nbeats_val),   batch_size=BATCH_SIZE, shuffle=False)
test_nbeats  = DataLoader(SerieDataset(X_nbeats_test,  y_nbeats_test),  batch_size=BATCH_SIZE, shuffle=False)

# Loader multivariante — LSTM, BiGRU-KAN, BNN
train_loader = DataLoader(SerieDataset(X_train_seq, y_train_seq), batch_size=BATCH_SIZE, shuffle=False)
val_loader   = DataLoader(SerieDataset(X_val_seq,   y_val_seq),   batch_size=BATCH_SIZE, shuffle=False)
test_loader  = DataLoader(SerieDataset(X_test_seq,  y_test_seq),  batch_size=BATCH_SIZE, shuffle=False)

# Construccion del loader mixto para NBEATSx
# X_price: precio historico normalizado (window,)
# X_exog:  exogenas de las proximas 24h normalizadas (horizon, n_features)
def crear_ventanas_mixtas(y_norm, X_norm, window=WINDOW, horizon=HORIZON):
    X_price, X_exog, y_out = [], [], []
    for i in range(len(y_norm) - window - horizon + 1):
        X_price.append(y_norm[i:i+window])
        X_exog.append(X_norm[i+window:i+window+horizon])
        y_out.append(y_norm[i+window:i+window+horizon])
    return np.array(X_price), np.array(X_exog), np.array(y_out)

X_price_train, X_exog_train, y_mix_train = crear_ventanas_mixtas(y_train_norm, X_train)
X_price_val,   X_exog_val,   y_mix_val   = crear_ventanas_mixtas(y_val_norm,   X_val)
X_price_test,  X_exog_test,  y_mix_test  = crear_ventanas_mixtas(y_test_norm,  X_test)

train_mixto = DataLoader(SerieDatasetMixto(X_price_train, X_exog_train, y_mix_train), batch_size=BATCH_SIZE, shuffle=False)
val_mixto   = DataLoader(SerieDatasetMixto(X_price_val,   X_exog_val,   y_mix_val),   batch_size=BATCH_SIZE, shuffle=False)
test_mixto  = DataLoader(SerieDatasetMixto(X_price_test,  X_exog_test,  y_mix_test),  batch_size=BATCH_SIZE, shuffle=False)

print(f'Loader univariante  — X_price: {X_price_train.shape}')
print(f'Loader mixto        — X_exog:  {X_exog_train.shape}')
print(f'Loader multivariante — X_seq:  {X_train_seq.shape}')

# Funciones de entrenamiento
def entrenar_modelo(model, train_loader, val_loader, epochs=100, lr=0.001, patience=10):
    model = model.to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = nn.MSELoss()
    best_val_loss = float('inf')
    patience_counter = 0
    best_state = None
    for epoch in range(epochs):
        model.train()
        for X_batch, y_batch in train_loader:
            X_batch, y_batch = X_batch.to(DEVICE), y_batch.to(DEVICE)
            optimizer.zero_grad()
            criterion(model(X_batch), y_batch).backward()
            optimizer.step()
        model.eval()
        val_loss = 0
        with torch.no_grad():
            for X_batch, y_batch in val_loader:
                X_batch, y_batch = X_batch.to(DEVICE), y_batch.to(DEVICE)
                val_loss += criterion(model(X_batch), y_batch).item()
        val_loss /= len(val_loader)
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= patience:
                break
    model.load_state_dict(best_state)
    return model, best_val_loss

def entrenar_modelo_mixto(model, train_loader, val_loader, epochs=100, lr=0.001, patience=10):
    """Funcion de entrenamiento para modelos con loader mixto (precio + exogenas)."""
    model = model.to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = nn.MSELoss()
    best_val_loss = float('inf')
    patience_counter = 0
    best_state = None
    for epoch in range(epochs):
        model.train()
        for X_price, X_exog, y_batch in train_loader:
            X_price, X_exog, y_batch = X_price.to(DEVICE), X_exog.to(DEVICE), y_batch.to(DEVICE)
            optimizer.zero_grad()
            criterion(model(X_price, X_exog), y_batch).backward()
            optimizer.step()
        model.eval()
        val_loss = 0
        with torch.no_grad():
            for X_price, X_exog, y_batch in val_loader:
                X_price, X_exog, y_batch = X_price.to(DEVICE), X_exog.to(DEVICE), y_batch.to(DEVICE)
                val_loss += criterion(model(X_price, X_exog), y_batch).item()
        val_loss /= len(val_loader)
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= patience:
                break
    model.load_state_dict(best_state)
    return model, best_val_loss

def predecir(model, loader):
    model.eval()
    preds = []
    with torch.no_grad():
        for X_batch, _ in loader:
            preds.append(model(X_batch.to(DEVICE)).cpu().numpy())
    return np.vstack(preds)

def predecir_mixto(model, loader):
    """Funcion de prediccion para modelos con loader mixto."""
    model.eval()
    preds = []
    with torch.no_grad():
        for X_price, X_exog, _ in loader:
            preds.append(model(X_price.to(DEVICE), X_exog.to(DEVICE)).cpu().numpy())
    return np.vstack(preds)

def generar_predicciones_secuenciales(preds_norm, y_seq):
    """Reorganiza predicciones por dias de 24h sin solapamiento."""
    indices = range(0, len(preds_norm), HORIZON)
    preds, reales = [], []
    for i in indices:
        if i < len(preds_norm):
            preds.append(desnormalizar(preds_norm[i]))
            reales.append(desnormalizar(y_seq[i]))
    return np.array(preds).flatten(), np.array(reales).flatten()

def gridsearch_modelo(nombre, create_fn, train_loader, val_loader, test_loader, y_test_seq):
    """
    GridSearch completo para modelos neuronales con recuperacion automatica.
    Si las predicciones ya existen en Drive, las carga directamente.
    Si el GridSearch se interrumpio, retoma desde donde lo dejo.
    """
    nombre_key   = nombre.lower().replace('-', '_').replace(' ', '_')
    MODEL_PATH   = f'{DRIVE_FOLDER}/{nombre_key}_model.pth'
    PARAMS_PATH  = f'{DRIVE_FOLDER}/{nombre_key}_best_params.pkl'
    HISTORY_PATH = f'{DRIVE_FOLDER}/{nombre_key}_search_history.pkl'
    PREDS_PATH   = f'{DRIVE_FOLDER}/preds_{nombre_key}.csv'

    if os.path.exists(PREDS_PATH):
        print(f'Predicciones {nombre} cargadas desde Drive')
        df_p = pd.read_csv(PREDS_PATH)
        preds_col = nombre if nombre in df_p.columns else df_p.columns[1]
        preds = df_p[preds_col].values
        return preds, y_test_real[:len(preds)]

    history = []
    if os.path.exists(HISTORY_PATH):
        with open(HISTORY_PATH, 'rb') as f:
            history = pickle.load(f)
        print(f'{nombre}: {len(history)} combinaciones ya probadas, retomando...')

    best_val_loss = min([h['val_loss'] for h in history]) if history else float('inf')
    best_params   = None
    if os.path.exists(PARAMS_PATH):
        with open(PARAMS_PATH, 'rb') as f:
            best_params = pickle.load(f)

    probadas   = [h['params'] for h in history]
    pendientes = [c for c in ALL_COMBINATIONS if c not in probadas]

    if pendientes:
        print(f'Iniciando GridSearch {nombre} ({len(pendientes)} combinaciones restantes de {len(ALL_COMBINATIONS)})...')
        for params in pendientes:
            iter_num = len(history) + 1
            print(f'  Iter {iter_num}/{len(ALL_COMBINATIONS)} | {params}')
            model = create_fn(**params)
            _, val_loss = entrenar_modelo(model, train_loader, val_loader, lr=params['lr'])
            print(f'    Val loss: {val_loss:.6f}')
            history.append({'params': params, 'val_loss': val_loss})
            with open(HISTORY_PATH, 'wb') as f:
                pickle.dump(history, f)
            if val_loss < best_val_loss:
                best_val_loss = val_loss
                best_params   = params
                print(f'    Nuevo mejor modelo')
                torch.save(model.state_dict(), MODEL_PATH)
                with open(PARAMS_PATH, 'wb') as f:
                    pickle.dump(best_params, f)
        print(f'\nGridSearch {nombre} completado')

    history_df = pd.DataFrame([{
        'hidden_size': h['params']['hidden_size'],
        'num_layers':  h['params']['num_layers'],
        'dropout':     h['params']['dropout'],
        'lr':          h['params']['lr'],
        'val_loss':    h['val_loss']
    } for h in history]).sort_values('val_loss')
    print(f'\nTop 10 configuraciones {nombre}:')
    print(history_df.head(10).to_string(index=False))
    history_df.to_csv(f'{DRIVE_FOLDER}/{nombre_key}_gridsearch_results.csv', index=False)

    print(f'\nMejores parametros: {best_params}')
    best_model = create_fn(**best_params).to(DEVICE)
    best_model.load_state_dict(torch.load(MODEL_PATH, map_location=DEVICE))
    preds_norm = predecir(best_model, test_loader)
    preds, reales = generar_predicciones_secuenciales(preds_norm, y_test_seq)

    df_out = pd.DataFrame({'datetime': df_test['datetime'].values[:len(preds)],
                           nombre: preds, 'y_real': reales})
    df_out.to_csv(PREDS_PATH, index=False, float_format='%.6f')
    print(f'Predicciones {nombre} guardadas')

    df_out.to_csv(f'/content/preds_{nombre_key}.csv', index=False, float_format='%.6f')
    from google.colab import files
    files.download(f'/content/preds_{nombre_key}.csv')

    return preds, reales

def gridsearch_modelo_mixto(nombre, create_fn, train_loader, val_loader, test_loader, y_test_seq):
    """
    GridSearch para modelos con loader mixto (NBEATSx).
    Identico a gridsearch_modelo pero usa entrenar_modelo_mixto y predecir_mixto.
    """
    nombre_key   = nombre.lower().replace('-', '_').replace(' ', '_')
    MODEL_PATH   = f'{DRIVE_FOLDER}/{nombre_key}_model.pth'
    PARAMS_PATH  = f'{DRIVE_FOLDER}/{nombre_key}_best_params.pkl'
    HISTORY_PATH = f'{DRIVE_FOLDER}/{nombre_key}_search_history.pkl'
    PREDS_PATH   = f'{DRIVE_FOLDER}/preds_{nombre_key}.csv'

    if os.path.exists(PREDS_PATH):
        print(f'Predicciones {nombre} cargadas desde Drive')
        df_p = pd.read_csv(PREDS_PATH)
        preds_col = nombre if nombre in df_p.columns else df_p.columns[1]
        preds = df_p[preds_col].values
        return preds, y_test_real[:len(preds)]

    history = []
    if os.path.exists(HISTORY_PATH):
        with open(HISTORY_PATH, 'rb') as f:
            history = pickle.load(f)
        print(f'{nombre}: {len(history)} combinaciones ya probadas, retomando...')

    best_val_loss = min([h['val_loss'] for h in history]) if history else float('inf')
    best_params   = None
    if os.path.exists(PARAMS_PATH):
        with open(PARAMS_PATH, 'rb') as f:
            best_params = pickle.load(f)

    probadas   = [h['params'] for h in history]
    pendientes = [c for c in ALL_COMBINATIONS if c not in probadas]

    if pendientes:
        print(f'Iniciando GridSearch {nombre} ({len(pendientes)} combinaciones restantes de {len(ALL_COMBINATIONS)})...')
        for params in pendientes:
            iter_num = len(history) + 1
            print(f'  Iter {iter_num}/{len(ALL_COMBINATIONS)} | {params}')
            model = create_fn(**params)
            _, val_loss = entrenar_modelo_mixto(model, train_loader, val_loader, lr=params['lr'])
            print(f'    Val loss: {val_loss:.6f}')
            history.append({'params': params, 'val_loss': val_loss})
            with open(HISTORY_PATH, 'wb') as f:
                pickle.dump(history, f)
            if val_loss < best_val_loss:
                best_val_loss = val_loss
                best_params   = params
                print(f'    Nuevo mejor modelo')
                torch.save(model.state_dict(), MODEL_PATH)
                with open(PARAMS_PATH, 'wb') as f:
                    pickle.dump(best_params, f)
        print(f'\nGridSearch {nombre} completado')

    history_df = pd.DataFrame([{
        'hidden_size': h['params']['hidden_size'],
        'num_layers':  h['params']['num_layers'],
        'dropout':     h['params']['dropout'],
        'lr':          h['params']['lr'],
        'val_loss':    h['val_loss']
    } for h in history]).sort_values('val_loss')
    print(f'\nTop 10 configuraciones {nombre}:')
    print(history_df.head(10).to_string(index=False))
    history_df.to_csv(f'{DRIVE_FOLDER}/{nombre_key}_gridsearch_results.csv', index=False)

    print(f'\nMejores parametros: {best_params}')
    best_model = create_fn(**best_params).to(DEVICE)
    best_model.load_state_dict(torch.load(MODEL_PATH, map_location=DEVICE))
    preds_norm = predecir_mixto(best_model, test_loader)
    preds, reales = generar_predicciones_secuenciales(preds_norm, y_test_seq)

    df_out = pd.DataFrame({'datetime': df_test['datetime'].values[:len(preds)],
                           nombre: preds, 'y_real': reales})
    df_out.to_csv(PREDS_PATH, index=False, float_format='%.6f')
    print(f'Predicciones {nombre} guardadas')

    df_out.to_csv(f'/content/preds_{nombre_key}.csv', index=False, float_format='%.6f')
    from google.colab import files
    files.download(f'/content/preds_{nombre_key}.csv')

    return preds, reales

print('Funciones definidas')
print(f'  Loader univariante:   X_price {X_price_train.shape}')
print(f'  Loader mixto:         X_exog  {X_exog_train.shape}')
print(f'  Loader multivariante: X_seq   {X_train_seq.shape}')

Loader univariante  — X_price: (34582, 168)
Loader mixto        — X_exog:  (34582, 24, 25)
Loader multivariante — X_seq:  (34582, 168, 25)
Funciones definidas
  Loader univariante:   X_price (34582, 168)
  Loader mixto:         X_exog  (34582, 24, 25)
  Loader multivariante: X_seq   (34582, 168, 25)


---
## 2. SARIMA — Baseline estadístico
Modelo SARIMAX con orden (2,1,0)(2,0,0,24) seleccionado mediante criterio AIC.
La prediccion se realiza de forma deslizante: cada dia se reajusta el modelo
sobre una ventana de los ultimos 30 dias para capturar cambios en la dinamica del mercado.
Referencia: Weron (2014)

In [8]:
from statsmodels.tsa.statespace.sarimax import SARIMAX

SARIMA_PREDS_PATH = f'{DRIVE_FOLDER}/preds_sarima.csv'

y_sarima_train = df_train[TARGET].values
y_sarima_val   = df_val[TARGET].values
y_sarima_test  = df_test[TARGET].values

if os.path.exists(SARIMA_PREDS_PATH):
    print('Predicciones SARIMA cargadas desde Drive')
    sarima_preds = pd.read_csv(SARIMA_PREDS_PATH)['SARIMA'].values
else:
    history_sarima = list(np.concatenate([y_sarima_train, y_sarima_val]))
    n_days = len(y_sarima_test) // HORIZON
    print(f'Generando predicciones SARIMA: {n_days} dias con ventana deslizante de 30 dias...')

    sarima_preds = []
    for i in range(n_days):
        # Ventana deslizante: ultimos 30 dias (720 horas)
        ventana = history_sarima[-720:]
        model = SARIMAX(ventana, order=(2,1,0), seasonal_order=(2,0,0,24))
        fit   = model.fit(disp=False)
        pred  = fit.forecast(steps=HORIZON)
        sarima_preds.extend(pred)
        history_sarima.extend(y_sarima_test[i*HORIZON:(i+1)*HORIZON])
        if i % 30 == 0:
            print(f'  Dia {i}/{n_days}')

    sarima_preds = np.array(sarima_preds).flatten()

    df_sarima = pd.DataFrame({
        'datetime': df_test['datetime'].values[:len(sarima_preds)],
        'SARIMA': sarima_preds
    })
    df_sarima.to_csv(SARIMA_PREDS_PATH, index=False, float_format='%.6f')
    df_sarima.to_csv('/content/preds_sarima.csv', index=False, float_format='%.6f')
    from google.colab import files
    files.download('/content/preds_sarima.csv')
    print('Guardado en Drive OK')

y_test_sarima_real = y_sarima_test[:len(sarima_preds)]
resultados['SARIMA'] = calcular_metricas(y_test_sarima_real, sarima_preds, 'SARIMA')

Predicciones SARIMA cargadas desde Drive
SARIMA: MAE=19.8187 | RMSE=27.9414 | MAPE=40.4859%


---
## 3. XGBoost — Baseline de machine learning
Un modelo independiente por cada hora del dia (24 modelos).
GridSearch sobre 81 combinaciones de hiperparametros.
Las 25 variables exogenas se usan como features.
Referencia: Jensen et al. (2024)

In [9]:
XGB_PREDS_PATH   = f'{DRIVE_FOLDER}/preds_xgboost.csv'
XGB_HISTORY_PATH = f'{DRIVE_FOLDER}/xgboost_search_history.pkl'

xgb_param_grid = {
    'n_estimators':  [100, 200, 500],
    'max_depth':     [3, 5, 8],
    'learning_rate': [0.01, 0.1, 0.3],
    'subsample':     [0.6, 0.8, 1.0]
}
XGB_COMBINATIONS = [
    {'n_estimators': n, 'max_depth': d, 'learning_rate': l, 'subsample': s}
    for n, d, l, s in itertools.product(
        xgb_param_grid['n_estimators'], xgb_param_grid['max_depth'],
        xgb_param_grid['learning_rate'], xgb_param_grid['subsample']
    )
]
print(f'Combinaciones XGBoost por hora: {len(XGB_COMBINATIONS)}')

if os.path.exists(XGB_PREDS_PATH):
    print('Predicciones XGBoost cargadas desde Drive')
    xgb_preds_test = pd.read_csv(XGB_PREDS_PATH)['XGBoost'].values
else:
    print('Entrenando 24 modelos XGBoost con GridSearch...')
    xgb_models      = {}
    xgb_best_params = {}
    xgb_results     = []

    for hora in range(HORIZON):
        XGB_HORA_PATH = f'{DRIVE_FOLDER}/xgb_hora_{hora}.pkl'

        if os.path.exists(XGB_HORA_PATH):
            xgb_models[hora] = joblib.load(XGB_HORA_PATH)
            if hora % 6 == 0:
                print(f'  Hora {hora:02d} cargada desde Drive')
        else:
            X_h_train = scaler_X.transform(df_train.loc[df_train['hour'] == hora, FEATURES])
            y_h_train = df_train.loc[df_train['hour'] == hora, TARGET].values
            X_h_val   = scaler_X.transform(df_val.loc[df_val['hour'] == hora, FEATURES])
            y_h_val   = df_val.loc[df_val['hour'] == hora, TARGET].values

            best_mae, best_model, best_p = float('inf'), None, None
            for params in XGB_COMBINATIONS:
                model = XGBRegressor(**params, random_state=42, n_jobs=-1)
                model.fit(X_h_train, y_h_train)
                mae = np.mean(np.abs(y_h_val - model.predict(X_h_val)))
                if mae < best_mae:
                    best_mae, best_model, best_p = mae, model, params

            xgb_models[hora]      = best_model
            xgb_best_params[hora] = best_p
            joblib.dump(best_model, XGB_HORA_PATH)
            xgb_results.append({'hora': hora, **best_p, 'val_mae': best_mae})
            if hora % 6 == 0:
                print(f'  Hora {hora:02d} | Mejor params: {best_p} | Val MAE: {best_mae:.4f}')

    pd.DataFrame(xgb_results).to_csv(f'{DRIVE_FOLDER}/xgboost_gridsearch_results.csv', index=False)

    xgb_preds_test = np.zeros(len(df_test))
    for hora in range(HORIZON):
        idx_hora = df_test['hour'] == hora
        X_h_test = scaler_X.transform(df_test.loc[idx_hora, FEATURES])
        xgb_preds_test[idx_hora.values] = xgb_models[hora].predict(X_h_test)

    df_xgb = pd.DataFrame({
        'datetime': df_test['datetime'].values,
        'XGBoost': xgb_preds_test
    })
    df_xgb.to_csv(XGB_PREDS_PATH, index=False, float_format='%.6f')
    df_xgb.to_csv('/content/preds_xgboost.csv', index=False, float_format='%.6f')
    from google.colab import files
    files.download('/content/preds_xgboost.csv')
    print('Guardado en Drive OK')

resultados['XGBoost'] = calcular_metricas(y_test_real, xgb_preds_test, 'XGBoost')

Combinaciones XGBoost por hora: 81
Predicciones XGBoost cargadas desde Drive
XGBoost: MAE=7.7267 | RMSE=11.5657 | MAPE=15.7097%


---
## 4. LSTM — Red recurrente multivariante
Arquitectura LSTM con las 25 variables exogenas como entrada.
Ventana de 168 horas (7 dias) para predecir las siguientes 24 horas.
GridSearch sobre 36 combinaciones de hiperparametros.
Referencia: Lago et al. (2018)

In [10]:
class LSTMModel(nn.Module):
    def __init__(self, hidden_size=128, num_layers=2, dropout=0.2, lr=0.001):
        super().__init__()
        self.lstm = nn.LSTM(input_size=N_FEATURES, hidden_size=hidden_size,
                            num_layers=num_layers,
                            dropout=dropout if num_layers > 1 else 0,
                            batch_first=True)
        self.fc = nn.Linear(hidden_size, HORIZON)
    def forward(self, x):
        out, _ = self.lstm(x)
        return self.fc(out[:, -1, :])

lstm_preds, y_lstm_real = gridsearch_modelo(
    nombre='LSTM',
    create_fn=LSTMModel,
    train_loader=train_loader,
    val_loader=val_loader,
    test_loader=test_loader,
    y_test_seq=y_test_seq
)
resultados['LSTM'] = calcular_metricas(y_lstm_real, lstm_preds, 'LSTM')

Predicciones LSTM cargadas desde Drive
LSTM: MAE=33.1358 | RMSE=41.8881 | MAPE=68.7626%


---
## 5. N-BEATS — Red de expansión de bases univariante
Implementacion propia en PyTorch siguiendo la arquitectura original de Oreshkin et al. (2020).
Se implementa en su configuracion univariante (solo precio historico) para respetar
el diseno original del modelo y servir como referencia de ablation frente a NBEATSx.
Referencia: Oreshkin et al. (2020)

In [11]:
class NBeatsBlock(nn.Module):
    def __init__(self, input_size, hidden_size, output_size, n_layers=4):
        super().__init__()
        layers = [nn.Linear(input_size, hidden_size), nn.ReLU()]
        for _ in range(n_layers - 1):
            layers += [nn.Linear(hidden_size, hidden_size), nn.ReLU()]
        self.fc_stack = nn.Sequential(*layers)
        self.backcast = nn.Linear(hidden_size, input_size)
        self.forecast = nn.Linear(hidden_size, output_size)
    def forward(self, x):
        h = self.fc_stack(x)
        return self.backcast(h), self.forecast(h)

class NBeatsModel(nn.Module):
    def __init__(self, hidden_size=256, num_layers=2, dropout=0.2, lr=0.001,
                 n_stacks=2, n_blocks=3):
        super().__init__()
        self.blocks = nn.ModuleList([
            NBeatsBlock(WINDOW, hidden_size, HORIZON, n_layers=4)
            for _ in range(n_stacks * n_blocks)
        ])
    def forward(self, x):
        residual = x
        forecast = torch.zeros(x.shape[0], HORIZON).to(x.device)
        for block in self.blocks:
            backcast, block_forecast = block(residual)
            residual = residual - backcast
            forecast = forecast + block_forecast
        return forecast

nbeats_preds, y_nbeats_real = gridsearch_modelo(
    nombre='N-BEATS',
    create_fn=NBeatsModel,
    train_loader=train_nbeats,
    val_loader=val_nbeats,
    test_loader=test_nbeats,
    y_test_seq=y_nbeats_test
)
resultados['N-BEATS'] = calcular_metricas(y_nbeats_real, nbeats_preds, 'N-BEATS')

Predicciones N-BEATS cargadas desde Drive
N-BEATS: MAE=28.8851 | RMSE=36.4748 | MAPE=56.2152%


  ---
  ## 6. NBEATSx — Extension multivariante de N-BEATS
  NBEATSx extiende N-BEATS incorporando variables exogenas como entrada adicional.
  Se implementa mediante la libreria NeuralForecast (Olivares et al., 2023),
  que proporciona una implementacion oficial y optimizada del modelo.
  Las 25 variables exogenas del dataset se usan como covariables historicas.
  El split temporal y el periodo de test son identicos al resto de modelos.
  Referencia: Olivares et al. (2023)

Este es la imprementacion con el paquete NeuralForecast, al final se hizo una implementacion desde cero con pytorch del modelo nbeatsx que aparece mas en la siguiente celda


In [12]:
'''
from neuralforecast import NeuralForecast
from neuralforecast.models import NBEATSx as NBEATSxModel
from neuralforecast.losses.pytorch import MAE as NF_MAE

NBEATSX_PREDS_PATH = f'{DRIVE_FOLDER}/preds_nbeatsx.csv'

if os.path.exists(NBEATSX_PREDS_PATH):
    print('Predicciones NBEATSx cargadas desde Drive')
    df_nbx = pd.read_csv(NBEATSX_PREDS_PATH)
    nbeatsx_preds  = df_nbx['NBEATSx'].values
    y_nbeatsx_real = y_test_real[:len(nbeatsx_preds)]
else:
    def preparar_df_nf(df_split, features=FEATURES, target=TARGET):
        df_nf = df_split[['datetime', target] + features].copy()
        df_nf = df_nf.rename(columns={'datetime': 'ds', target: 'y'})
        df_nf['unique_id'] = 'spain'
        df_nf['ds'] = pd.to_datetime(df_nf['ds']).dt.tz_localize(None)
        return df_nf

    df_train_nf    = preparar_df_nf(df_train)
    df_val_nf      = preparar_df_nf(df_val)
    df_test_nf     = preparar_df_nf(df_test)
    df_trainval_nf = pd.concat([df_train_nf, df_val_nf]).reset_index(drop=True)

    print(f'Dataset NeuralForecast train+val: {len(df_trainval_nf)} registros')
    print(f'Dataset NeuralForecast test: {len(df_test_nf)} registros')

    nbx_param_grid = []
    for n_blocks, stack_types in [
        ([3, 3],       ['identity', 'identity']),
        ([4, 4],       ['identity', 'identity']),
        ([5, 5],       ['identity', 'identity']),
        ([6, 6],       ['identity', 'identity']),
        ([2, 2, 2],    ['identity', 'identity', 'identity']),
        ([3, 3, 3],    ['identity', 'identity', 'identity']),
        ([4, 4, 4],    ['identity', 'identity', 'identity']),
        ([3, 3, 3, 3], ['identity', 'identity', 'identity', 'identity']),
        ([3, 3],       ['trend', 'seasonality']),
    ]:
        for mlp_size in [128, 256, 512]:
            for lr in [1e-3, 1e-4]:
                nbx_param_grid.append({
                    'n_blocks':    n_blocks,
                    'stack_types': stack_types,
                    'mlp_units':   [[mlp_size, mlp_size]] * len(n_blocks),
                    'learning_rate': lr
                })

    seen, nbx_param_grid_clean = [], []
    for c in nbx_param_grid:
        if c not in seen:
            seen.append(c)
            nbx_param_grid_clean.append(c)
    nbx_param_grid = nbx_param_grid_clean
    print(f'GridSearch NBEATSx: {len(nbx_param_grid)} configuraciones...')

    best_val_mae    = float('inf')
    best_nbx_config = None
    nbx_results     = []

    val_cutoff   = df_trainval_nf['ds'].max() - pd.Timedelta(days=365)
    df_train_nbx = df_trainval_nf[df_trainval_nf['ds'] <= val_cutoff].copy()
    df_val_nbx   = df_trainval_nf[df_trainval_nf['ds'] >  val_cutoff].copy()

    for i, config in enumerate(nbx_param_grid):
        print(f'  Config {i+1}/{len(nbx_param_grid)}: {config}')
        model = NBEATSxModel(
            h=HORIZON,
            input_size=WINDOW,
            hist_exog_list=FEATURES,
            max_steps=500,
            early_stop_patience_steps=10,
            val_check_steps=50,
            loss=NF_MAE(),
            **config
        )
        nf = NeuralForecast(models=[model], freq='h')
        nf.fit(df=df_train_nbx, val_size=HORIZON * 30)

        preds_val = nf.predict(df=df_train_nbx)
        y_val_nbx = df_train_nbx['y'].values[-len(preds_val):]
        val_mae   = np.mean(np.abs(y_val_nbx - preds_val['NBEATSx'].values))
        print(f'    Val MAE: {val_mae:.4f}')
        nbx_results.append({'config': i, 'val_mae': val_mae})

        if val_mae < best_val_mae:
            best_val_mae    = val_mae
            best_nbx_config = config
            print(f'    Nuevo mejor modelo')

    pd.DataFrame(nbx_results).to_csv(f'{DRIVE_FOLDER}/nbeatsx_gridsearch_results.csv', index=False)
    print(f'\nMejor configuracion: {best_nbx_config} | Val MAE: {best_val_mae:.4f}')

    print('\nReentrenando NBEATSx con train+val completo...')
    model_final = NBEATSxModel(
        h=HORIZON,
        input_size=WINDOW,
        hist_exog_list=FEATURES,
        max_steps=1000,
        early_stop_patience_steps=15,
        val_check_steps=50,
        loss=NF_MAE(),
        **best_nbx_config
    )
    nf_final = NeuralForecast(models=[model_final], freq='h')
    nf_final.fit(df=df_trainval_nf, val_size=HORIZON * 30)
    nf_final.save(f'{DRIVE_FOLDER}/nbeatsx_model', overwrite=True)

    # Prediccion rolling dia a dia sobre el test
    # NeuralForecast con hist_exog_list solo predice el primer horizonte.
    # Se itera dia a dia anadiendo cada dia al contexto historico.
    print('\nGenerando predicciones sobre test (rolling window dia a dia)...')
    nbeatsx_preds = []
    df_contexto   = df_trainval_nf.copy()
    n_dias_test   = len(df_test_nf) // HORIZON

    for i in range(n_dias_test):
        df_dia = df_test_nf.iloc[i*HORIZON:(i+1)*HORIZON].copy()
        df_input = pd.concat([df_contexto, df_dia]).reset_index(drop=True)

        try:
            preds = nf_final.predict(df=df_input)
            nbeatsx_preds.extend(preds['NBEATSx'].values[:HORIZON])
        except Exception as e:
            print(f'  Dia {i} error: {e}')
            nbeatsx_preds.extend([np.nan] * HORIZON)

        df_contexto = df_input.copy()

        if i % 30 == 0:
            print(f'  Dia {i}/{n_dias_test}')

    nbeatsx_preds  = np.array(nbeatsx_preds).flatten()
    y_nbeatsx_real = y_test_real[:len(nbeatsx_preds)]
    print(f'Predicciones generadas: {len(nbeatsx_preds)}')

    df_nbx_out = pd.DataFrame({
        'datetime': df_test_nf['ds'].values[:len(nbeatsx_preds)],
        'NBEATSx':  nbeatsx_preds,
        'y_real':   y_nbeatsx_real
    })
    df_nbx_out.to_csv(NBEATSX_PREDS_PATH, index=False, float_format='%.6f')
    df_nbx_out.to_csv('/content/preds_nbeatsx.csv', index=False, float_format='%.6f')
    from google.colab import files
    files.download('/content/preds_nbeatsx.csv')
    print('Guardado en Drive OK')

resultados['NBEATSx'] = calcular_metricas(y_nbeatsx_real, nbeatsx_preds, 'NBEATSx')
'''

"\nfrom neuralforecast import NeuralForecast\nfrom neuralforecast.models import NBEATSx as NBEATSxModel\nfrom neuralforecast.losses.pytorch import MAE as NF_MAE\n\nNBEATSX_PREDS_PATH = f'{DRIVE_FOLDER}/preds_nbeatsx.csv'\n\nif os.path.exists(NBEATSX_PREDS_PATH):\n    print('Predicciones NBEATSx cargadas desde Drive')\n    df_nbx = pd.read_csv(NBEATSX_PREDS_PATH)\n    nbeatsx_preds  = df_nbx['NBEATSx'].values\n    y_nbeatsx_real = y_test_real[:len(nbeatsx_preds)]\nelse:\n    def preparar_df_nf(df_split, features=FEATURES, target=TARGET):\n        df_nf = df_split[['datetime', target] + features].copy()\n        df_nf = df_nf.rename(columns={'datetime': 'ds', target: 'y'})\n        df_nf['unique_id'] = 'spain'\n        df_nf['ds'] = pd.to_datetime(df_nf['ds']).dt.tz_localize(None)\n        return df_nf\n\n    df_train_nf    = preparar_df_nf(df_train)\n    df_val_nf      = preparar_df_nf(df_val)\n    df_test_nf     = preparar_df_nf(df_test)\n    df_trainval_nf = pd.concat([df_train_nf, df

Modelo N-Beatsx con pytorch

In [13]:
class NBeatsXModel(nn.Module):
    """
    NBEATSx en PyTorch con loader mixto.
    Entrada: precio historico (window,) + exogenas de las proximas 24h (horizon, n_features)
    Las exogenas se proyectan y concatenan al precio antes de cada bloque.
    Referencia: Olivares et al. (2023)
    """
    def __init__(self, hidden_size=256, num_layers=2, dropout=0.2, lr=0.001,
                 n_stacks=2, n_blocks=3):
        super().__init__()
        exog_flat = HORIZON * N_FEATURES
        self.exog_proj = nn.Linear(exog_flat, WINDOW)
        self.blocks = nn.ModuleList([
            NBeatsBlock(WINDOW * 2, hidden_size, HORIZON, n_layers=4)
            for _ in range(n_stacks * n_blocks)
        ])
        self.drop = nn.Dropout(dropout)

    def forward(self, x_price, x_exog):
        # x_price: (batch, window)
        # x_exog:  (batch, horizon, n_features)
        batch = x_price.shape[0]
        x_exog_flat = x_exog.reshape(batch, -1)
        x_exog_proj = self.exog_proj(x_exog_flat)
        x = torch.cat([x_price, x_exog_proj], dim=-1)  # (batch, window*2)
        residual = x
        forecast = torch.zeros(batch, HORIZON).to(x.device)
        for block in self.blocks:
            backcast, block_forecast = block(residual)
            residual = residual - backcast
            forecast = forecast + block_forecast
        return forecast

nbeatsx_preds, y_nbeatsx_real = gridsearch_modelo_mixto(
    nombre='NBEATSx',
    create_fn=NBeatsXModel,
    train_loader=train_mixto,
    val_loader=val_mixto,
    test_loader=test_mixto,
    y_test_seq=y_mix_test
)
resultados['NBEATSx'] = calcular_metricas(y_nbeatsx_real, nbeatsx_preds, 'NBEATSx')

Predicciones NBEATSx cargadas desde Drive
NBEATSx: MAE=29.8953 | RMSE=40.5858 | MAPE=60.0224%


---
## 7. BiGRU-KAN — Red recurrente bidireccional con capas KAN
Arquitectura hibrida que combina una GRU bidireccional con capas
Kolmogorov-Arnold Network (KAN) para la prediccion final.
Las capas KAN aproximan funciones no lineales con mayor expresividad que las capas lineales.
GridSearch sobre 36 combinaciones de hiperparametros.
Referencia: Yang et al. (2025)

In [14]:
class KANLayer(nn.Module):
    def __init__(self, in_features, out_features):
        super().__init__()
        self.fc         = nn.Linear(in_features, out_features)
        self.activation = nn.SiLU()
        self.scale      = nn.Parameter(torch.ones(out_features))
    def forward(self, x):
        return self.activation(self.fc(x)) * self.scale

class BiGRUKANModel(nn.Module):
    def __init__(self, hidden_size=128, num_layers=2, dropout=0.2, lr=0.001,
                 kan_dim=64, output_size=24):
        super().__init__()
        self.bigru = nn.GRU(input_size=N_FEATURES, hidden_size=hidden_size,
                            num_layers=num_layers,
                            dropout=dropout if num_layers > 1 else 0,
                            batch_first=True, bidirectional=True)
        self.kan1   = KANLayer(hidden_size * 2, kan_dim)
        self.kan2   = KANLayer(kan_dim, kan_dim)
        self.fc_out = nn.Linear(kan_dim, output_size)
    def forward(self, x):
        out, _ = self.bigru(x)
        out = out[:, -1, :]
        out = self.kan1(out)
        out = self.kan2(out)
        return self.fc_out(out)

bigru_preds, y_bigru_real = gridsearch_modelo(
    nombre='BiGRU-KAN',
    create_fn=BiGRUKANModel,
    train_loader=train_loader,
    val_loader=val_loader,
    test_loader=test_loader,
    y_test_seq=y_test_seq
)
resultados['BiGRU-KAN'] = calcular_metricas(y_bigru_real, bigru_preds, 'BiGRU-KAN')

Predicciones BiGRU-KAN cargadas desde Drive
BiGRU-KAN: MAE=31.2293 | RMSE=39.7953 | MAPE=61.7218%


---
## 8. BNN con MC Dropout — Red bayesiana probabilistica
Un modelo independiente por cada hora del dia (24 modelos).
El dropout permanece activo en inferencia para generar distribucion de predicciones.
Se realizan 1000 pases estocásticos para estimar media y desviacion tipica.
GridSearch sobre 36 combinaciones x 24 horas = 864 entrenamientos.
Referencia: Das et al. (2025)

In [15]:
class BNNModel(nn.Module):
    def __init__(self, hidden_size=128, num_layers=2, dropout=0.2, lr=0.001):
        super().__init__()
        layers  = []
        in_size = N_FEATURES
        for _ in range(num_layers):
            layers += [nn.Linear(in_size, hidden_size), nn.ReLU(), nn.Dropout(dropout)]
            in_size = hidden_size
        layers.append(nn.Linear(hidden_size, 1))
        self.net = nn.Sequential(*layers)
    def forward(self, x):
        return self.net(x)
    def mc_predict(self, x, n_samples=1000):
        self.train()  # dropout activo en inferencia
        x = torch.FloatTensor(x).to(DEVICE)
        preds = []
        with torch.no_grad():
            for _ in range(n_samples):
                preds.append(self.forward(x).cpu().numpy())
        preds = np.array(preds).squeeze()
        return preds.mean(axis=0), preds.std(axis=0)

BNN_PREDS_PATH   = f'{DRIVE_FOLDER}/preds_bnn.csv'
BNN_HISTORY_PATH = f'{DRIVE_FOLDER}/bnn_search_history.pkl'
MC_SAMPLES       = 1000

if os.path.exists(BNN_PREDS_PATH):
    print('Predicciones BNN cargadas desde Drive')
    bnn_df    = pd.read_csv(BNN_PREDS_PATH)
    bnn_mu    = bnn_df['BNN_mu'].values
    bnn_lower = bnn_df['BNN_lower'].values
    bnn_upper = bnn_df['BNN_upper'].values
else:
    bnn_history = {}
    if os.path.exists(BNN_HISTORY_PATH):
        with open(BNN_HISTORY_PATH, 'rb') as f:
            bnn_history = pickle.load(f)
        print(f'BNN: historial cargado ({len(bnn_history)} horas completadas)')

    print(f'Iniciando GridSearch BNN (36 combinaciones x 24 horas = 864 entrenamientos)...')

    bnn_mu_norm    = np.zeros(len(df_test))
    bnn_sigma_norm = np.zeros(len(df_test))
    bnn_best_params = {}

    for hora in range(HORIZON):
        BNN_HORA_PATH = f'{DRIVE_FOLDER}/bnn_hora_{hora}.pth'

        X_h_train = scaler_X.transform(df_train.loc[df_train['hour'] == hora, FEATURES])
        y_h_train = scaler_y.transform(df_train.loc[df_train['hour'] == hora, [TARGET]]).flatten()
        X_h_val   = scaler_X.transform(df_val.loc[df_val['hour'] == hora, FEATURES])
        y_h_val   = scaler_y.transform(df_val.loc[df_val['hour'] == hora, [TARGET]]).flatten()

        train_h = DataLoader(SerieDataset(X_h_train, y_h_train.reshape(-1,1)), batch_size=32, shuffle=False)
        val_h   = DataLoader(SerieDataset(X_h_val,   y_h_val.reshape(-1,1)),   batch_size=32, shuffle=False)

        if str(hora) in bnn_history and os.path.exists(BNN_HORA_PATH):
            best_p = bnn_history[str(hora)]['best_params']
            bnn = BNNModel(**best_p).to(DEVICE)
            bnn.load_state_dict(torch.load(BNN_HORA_PATH, map_location=DEVICE))
            if hora % 6 == 0:
                print(f'  Hora {hora:02d} cargada desde Drive')
        else:
            best_val_loss, best_p, best_bnn = float('inf'), None, None
            hora_results = []
            for params in ALL_COMBINATIONS:
                bnn = BNNModel(**params).to(DEVICE)
                _, val_loss = entrenar_modelo(bnn, train_h, val_h, lr=params['lr'])
                hora_results.append({'params': params, 'val_loss': val_loss})
                if val_loss < best_val_loss:
                    best_val_loss, best_p, best_bnn = val_loss, params, bnn
            torch.save(best_bnn.state_dict(), BNN_HORA_PATH)
            bnn_history[str(hora)] = {'best_params': best_p, 'best_val_loss': best_val_loss,
                                      'all_results': hora_results}
            with open(BNN_HISTORY_PATH, 'wb') as f:
                pickle.dump(bnn_history, f)
            if hora % 6 == 0:
                print(f'  Hora {hora:02d} | Best params: {best_p} | Val loss: {best_val_loss:.6f}')

        bnn_best_params[hora] = bnn_history[str(hora)]['best_params']
        idx_hora = df_test['hour'] == hora
        X_h_test = scaler_X.transform(df_test.loc[idx_hora, FEATURES])
        mu, sigma = bnn.mc_predict(X_h_test, n_samples=MC_SAMPLES)
        bnn_mu_norm[idx_hora.values]    = mu
        bnn_sigma_norm[idx_hora.values] = sigma

    bnn_mu    = desnormalizar(bnn_mu_norm.reshape(-1,1))
    bnn_sigma = bnn_sigma_norm * (scaler_y.data_max_[0] - scaler_y.data_min_[0]) / 2
    z_90      = 1.645
    bnn_lower = bnn_mu - z_90 * bnn_sigma
    bnn_upper = bnn_mu + z_90 * bnn_sigma

    # Tabla resumen GridSearch BNN
    bnn_results_flat = []
    for hora in range(HORIZON):
        if str(hora) in bnn_history:
            for r in bnn_history[str(hora)]['all_results']:
                bnn_results_flat.append({'hora': hora, **r['params'], 'val_loss': r['val_loss']})
    bnn_results_df = pd.DataFrame(bnn_results_flat)
    bnn_avg = bnn_results_df.groupby(['hidden_size','num_layers','dropout','lr'])['val_loss'].mean().reset_index()
    print('\nTop 10 configuraciones BNN (promedio sobre 24 horas):')
    print(bnn_avg.sort_values('val_loss').head(10).to_string(index=False))
    bnn_avg.to_csv(f'{DRIVE_FOLDER}/bnn_gridsearch_results.csv', index=False)

    df_bnn = pd.DataFrame({
        'datetime': df_test['datetime'].values[:len(bnn_mu)],
        'BNN_mu': bnn_mu, 'BNN_lower': bnn_lower, 'BNN_upper': bnn_upper
    })
    df_bnn.to_csv(BNN_PREDS_PATH, index=False, float_format='%.6f')
    df_bnn.to_csv('/content/preds_bnn.csv', index=False, float_format='%.6f')
    from google.colab import files
    files.download('/content/preds_bnn.csv')
    print('Guardado en Drive OK')

resultados['BNN'] = calcular_metricas(y_test_real, bnn_mu, 'BNN')
picp = np.mean((y_test_real >= bnn_lower) & (y_test_real <= bnn_upper))
mpiw = np.mean(bnn_upper - bnn_lower)
print(f'BNN PICP: {picp:.4f} | MPIW: {mpiw:.4f}')
resultados['BNN']['PICP'] = picp
resultados['BNN']['MPIW'] = mpiw

Predicciones BNN cargadas desde Drive
BNN: MAE=12.7812 | RMSE=16.2439 | MAPE=26.3550%
BNN PICP: 0.9448 | MPIW: 67.2799


---
## 9. Tabla comparativa y exportacion final

In [16]:
# Tabla comparativa de metricas predictivas
tabla = pd.DataFrame(resultados).T[['MAE', 'RMSE', 'MAPE']].round(4)
tabla.columns = ['MAE (EUR/MWh)', 'RMSE (EUR/MWh)', 'MAPE (%)']
tabla.index.name = 'Modelo'
print('\n=== TABLA COMPARATIVA DE METRICAS ===')
print(tabla.to_string())
tabla.to_csv(f'{DRIVE_FOLDER}/tabla_metricas.csv')
print('\nTabla guardada en Drive')


=== TABLA COMPARATIVA DE METRICAS ===
           MAE (EUR/MWh)  RMSE (EUR/MWh)  MAPE (%)
Modelo                                            
SARIMA           19.8187         27.9414   40.4859
XGBoost           7.7267         11.5657   15.7097
LSTM             33.1358         41.8881   68.7626
N-BEATS          28.8851         36.4748   56.2152
NBEATSx          29.8953         40.5858   60.0224
BiGRU-KAN        31.2293         39.7953   61.7218
BNN              12.7812         16.2439   26.3550

Tabla guardada en Drive


In [17]:
# Exportar CSV completo con todas las predicciones para el notebook de arbitraje
TODAS_PREDS_PATH = f'{DRIVE_FOLDER}/predicciones_test.csv'

n_comun = min(
    len(sarima_preds), len(xgb_preds_test),
    len(lstm_preds), len(nbeats_preds), len(nbeatsx_preds),
    len(bigru_preds), len(bnn_mu), len(y_test_real)
)

df_predicciones = pd.DataFrame({
    'datetime_test': df_test['datetime'].values[:n_comun],
    'precio_real':   y_test_real[:n_comun],
    'SARIMA':        sarima_preds[:n_comun],
    'XGBoost':       xgb_preds_test[:n_comun],
    'LSTM':          lstm_preds[:n_comun],
    'N-BEATS':       nbeats_preds[:n_comun],
    'NBEATSx':       nbeatsx_preds[:n_comun],
    'BiGRU-KAN':     bigru_preds[:n_comun],
    'BNN_mu':        bnn_mu[:n_comun],
    'BNN_lower':     bnn_lower[:n_comun],
    'BNN_upper':     bnn_upper[:n_comun]
})

df_predicciones.to_csv(TODAS_PREDS_PATH, index=False)
print(f'CSV completo guardado: {TODAS_PREDS_PATH}')
print(f'Registros: {len(df_predicciones)}')
print('\nListo para el notebook de arbitraje.')
df_predicciones.head()

CSV completo guardado: /content/drive/MyDrive/TFM/predicciones_test.csv
Registros: 9888

Listo para el notebook de arbitraje.


,datetime_test,precio_real,SARIMA,XGBoost,LSTM,N-BEATS,NBEATSx,BiGRU-KAN,BNN_mu,BNN_lower,BNN_upper
0,2025-01-01 00:00:00,131.59,129.640520,125.210762,68.312057,70.962760,54.646820,60.975548,118.260289,89.117647,147.402931
1,2025-01-01 01:00:00,131.49,128.121608,127.000938,68.400665,62.518166,51.421413,55.575832,133.205740,110.353861,156.057620
2,2025-01-01 02:00:00,131.42,125.737754,131.102356,63.260113,56.125744,51.491936,49.286201,116.807616,79.527600,154.087632
3,2025-01-01 03:00:00,120.49,122.303925,128.669235,64.032242,48.135277,55.611919,48.338951,91.878715,48.747828,135.009602
4,2025-01-01 04:00:00,112.30,125.278675,124.331032,68.762268,53.669384,64.787361,60.389534,107.744953,80.204212,135.285693
